In [23]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequences = [
    "I've been waiting for the Hugging Face course my whole life.",
    "This course is amazing"
]

batch = tokenizer(sequences, padding=True, truncation=True, return_tensors='pt')

# This is new
batch["labels"] = torch.tensor([1, 1])
optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
from datasets import load_dataset

raw_data = load_dataset("glue", "mrpc")
raw_data

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [25]:
raw_train_dataset = raw_data['train']
raw_train_dataset[5]

{'sentence1': 'Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 'sentence2': "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .",
 'label': 1,
 'idx': 5}

In [26]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [27]:
raw_data['train']['sentence1']

Column(['Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', "Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .", 'They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .', 'Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .', 'The stock rose $ 2.11 , or about 11 percent , to close Friday at $ 21.51 on the New York Stock Exchange .'])

In [28]:
def tokenize_elements(example):
    return tokenizer(example['sentence1'], example['sentence2'], truncation=True)

In [29]:
tokenized_dataset = raw_data.map(tokenize_elements, batched=True)
tokenized_dataset

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [30]:
# It is applied after tokenization and it will apply tokenization based on longest sentence and not based on max_length of model.
# It expect a tokenizer
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [31]:
# Let's practice DataCollator for understanding
samples = tokenized_dataset["train"][:8]
samples = {k:v for k,v in samples.items() if k not in ['sentence1', 'sentence2','idx'] }
[len(i) for i in samples['input_ids']]

[52, 59, 47, 69, 60, 50, 66, 32]

In [32]:
sample_batch = data_collator(samples)
{k:v.shape for k,v in sample_batch.items()}

{'input_ids': torch.Size([8, 69]),
 'token_type_ids': torch.Size([8, 69]),
 'attention_mask': torch.Size([8, 69]),
 'labels': torch.Size([8])}

In [33]:
from transformers import TrainingArguments
training_args = TrainingArguments("best-trained") # It will save the new model in following directory

In [34]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
# Simple training
from transformers import Trainer
trainer = Trainer(
    model,
    training_args,
    train_dataset= tokenized_dataset['train'],
    eval_dataset= tokenized_dataset['validation'],
    data_collator= data_collator,
    processing_class = tokenizer,
)

In [36]:
# It will start training the model
# trainer.train()

In [37]:
# !pip install evaluate

In [41]:
import evaluate
import numpy as np
from transformers import Trainer, TrainingArguments

# Compute metrics
def compute_metrics(eval_pred):
  metrics = evaluate.load('glue','mrpc')
  logits, labels = eval_pred.predictions, eval_pred.label_ids
  predications = np.argmax(logits, axis=1)
  return metrics.compute(predictions=predications, references=labels)

# Evaluation training
training_args = TrainingArguments(
    "best-trained",
    eval_strategy="epoch",
    fp16=True,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    )

# Trainer with compute metrics
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [42]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.674888,0.843137,0.886525
2,No log,0.800663,0.823529,0.877551
3,0.749284,0.879127,0.835784,0.884283


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=690, training_loss=0.6367137079653533, metrics={'train_runtime': 160.2958, 'train_samples_per_second': 68.648, 'train_steps_per_second': 4.305, 'total_flos': 389659250767680.0, 'train_loss': 0.6367137079653533, 'epoch': 3.0})